In [3]:
import pandas as pd
reduced_df = pd.read_feather('./processed/reduced_coerced_engagement_list.feather')

In [4]:
import shutil
from openpyxl import load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows

source_path = './raw/vector/Vector Engagements-01.21.2026_source.xlsx'
output_path = './review/vector/Vector Engagements-01.21.2026.xlsx'

# Read the original data
vector_engagements = pd.read_excel(source_path, sheet_name='Vector List Update')

# Display available columns
print("Available columns:", vector_engagements.columns.tolist())

# Find the actual hours column (case-insensitive search)
hours_col = None
for col in vector_engagements.columns:
    if 'actual' in col.lower() and 'hour' in col.lower():
        hours_col = col
        break

if hours_col is None:
    print("WARNING: Could not find ActualHours column")
    hours_col = 'Sum of ActualHours'  # fallback

# Store original counts for comparison
original_client_count = vector_engagements['Client'].nunique()
original_actualhours = vector_engagements[hours_col].sum() if hours_col in vector_engagements.columns else 0
original_row_count = len(vector_engagements)

print(f"\nOriginal file - Rows: {original_row_count}, Clients: {original_client_count}, {hours_col}: {original_actualhours}")
print(f"Original Eng ID duplicates: {vector_engagements['Eng ID'].duplicated().sum()}")
print(f"Reduced_df Engagement ID duplicates: {reduced_df['Engagement ID'].duplicated().sum()}")

# Deduplicate the reduced_df by Engagement ID, keeping first occurrence
reduced_df_dedup = reduced_df[['Engagement ID', 'Client ID', 'ANSR / Tech Revenue FYTD', 'ANSR / Tech Revenue ETD']].drop_duplicates(subset=['Engagement ID'], keep='first')

print(f"Reduced_df after dedup: {len(reduced_df_dedup)} rows")

# Check for duplicates in the original vector file that might cause issues
if vector_engagements['Eng ID'].duplicated().any():
    print("WARNING: Original vector file has duplicate Eng IDs!")
    # Build aggregation dict dynamically
    agg_dict = {'Client': 'first'}
    if hours_col and hours_col in vector_engagements.columns:
        agg_dict[hours_col] = 'sum'
    # Add all other columns
    for col in vector_engagements.columns:
        if col not in ['Eng ID', 'Client'] and (not hours_col or col != hours_col):
            agg_dict[col] = 'first'
    
    vector_engagements = vector_engagements.groupby('Eng ID', as_index=False).agg(agg_dict)
    print(f"After deduplication: {len(vector_engagements)} rows, {vector_engagements['Client'].nunique()} clients")

# Merge to add the new columns
vector_engagements = vector_engagements.merge(
    reduced_df_dedup, 
    left_on='Eng ID', 
    right_on='Engagement ID', 
    how='left'
)

# Drop the duplicate 'Engagement ID' column from the merge
vector_engagements = vector_engagements.drop(columns=['Engagement ID'])

# Calculate final counts
final_client_count = vector_engagements['Client'].nunique()
final_actualhours = vector_engagements[hours_col].sum() if hours_col in vector_engagements.columns else 0
final_row_count = len(vector_engagements)

# Create summary table
summary = pd.DataFrame({
    'Metric': ['Total Rows', 'Unique Clients', f'Total {hours_col}'],
    'Original': [original_row_count, original_client_count, original_actualhours],
    'Final': [final_row_count, final_client_count, final_actualhours],
    'Difference': [
        final_row_count - original_row_count,
        final_client_count - original_client_count,
        final_actualhours - original_actualhours
    ]
})

# Copy the entire source workbook to preserve all existing sheets
shutil.copy(source_path, output_path)
print(f"\nCopied source workbook to: {output_path}")

# Load the copied workbook and update/add sheets
wb = load_workbook(output_path)
print(f"Existing sheets: {wb.sheetnames}")

# Update the 'Vector List Update' sheet with modified data
if 'Vector List Update' in wb.sheetnames:
    del wb['Vector List Update']
ws_data = wb.create_sheet('Vector List Update', 0)  # Insert at the beginning

# Write dataframe to worksheet
for r_idx, row in enumerate(dataframe_to_rows(vector_engagements, index=False, header=True), 1):
    for c_idx, value in enumerate(row, 1):
        ws_data.cell(row=r_idx, column=c_idx, value=value)

# Add Summary sheet
if 'Summary' in wb.sheetnames:
    del wb['Summary']
ws_summary = wb.create_sheet('Summary')

for r_idx, row in enumerate(dataframe_to_rows(summary, index=False, header=True), 1):
    for c_idx, value in enumerate(row, 1):
        ws_summary.cell(row=r_idx, column=c_idx, value=value)

# Save the workbook
wb.save(output_path)
wb.close()

print(f"\nModified file saved to: {output_path}")
print(f"Final sheets: {load_workbook(output_path).sheetnames}")
print("\nSummary:")
print(summary.to_string(index=False))

Available columns: ['Eng ID', 'Client', 'Eng Name', 'CapCategory', 'Vector / Non Vector', 'Vector EP', 'Vector EM', 'Sum of ActualHours', 'NSR', 'CAP TER', 'Update Notes', 'GDS Scorecard Y/N']

Original file - Rows: 998, Clients: 512, Sum of ActualHours: 16041.699999999999
Original Eng ID duplicates: 0
Reduced_df Engagement ID duplicates: 0
Reduced_df after dedup: 210873 rows

Copied source workbook to: ./review/vector/Vector Engagements-01.21.2026.xlsx
Existing sheets: ['Vector Summary Totals', 'Vector List Summary by Client', 'Vector List Update']

Modified file saved to: ./review/vector/Vector Engagements-01.21.2026.xlsx
Final sheets: ['Vector List Update', 'Vector Summary Totals', 'Vector List Summary by Client', 'Summary']

Summary:
                  Metric  Original   Final  Difference
              Total Rows     998.0   998.0         0.0
          Unique Clients     512.0   512.0         0.0
Total Sum of ActualHours   16041.7 16041.7         0.0
